# 🔤 From Text to Attention

By the end of this notebook, you'll understand:
1. How text becomes tokens (and why it matters)
2. How tokens become meaningful vectors (embeddings)
3. How position information gets added
4. How attention actually works — coded from scratch

Let's start!

---
# Part 1: Tokenization
*What does the model actually see?*

---

In [ ]:
# First, install tiktoken (OpenAI's tokenizer)
%pip install tiktoken -q

In [ ]:
import tiktoken
import numpy as np

In [ ]:
tokenizer=tiktoken.get_encoding("o200k_base")
encode=tokenizer.encode("hello world")
print(encode)
assert tokenizer.decode(tokenizer.encode("hello world"))=="hello world"
# to get the tokeniser corresponding to a specific model in the openAI API:
enc=tiktoken.encoding_for_model("gpt-4o")
print(f"vocabulary size: {tokenizer.n_vocab:,}")

## 1.1 Basic Tokenization

Let's see how text gets converted to numbers.

In [ ]:
#simple example
text="hello world"
tokens=tokenizer.encode(text)
print(f"text:{text}")
print(f"Tokens:{tokens}")
print(f"number of tokens: {len(tokens)}")

#let's see what each token represents
print("\nToken Breakdown:")
for token in tokens:
    print(f"{token}->{tokenizer.decode([token])}")
    


Notice: "Hello" is one token, but " world" (with the space!) is another.
The space attaches to the following word.

In [ ]:
#let's try more interesting examples
examples = [
    "Hello world",
    "don't",
    "artificial intelligence",
    "I love AI",
    "supercalifragilisticexpialidocious",
    "🚀",
    "café",
    "    spaces    ",  # multiple spaces
]
print("how different texts get tokenized:\n")
for text in examples:
    tokens=tokenizer.encode(text)
    print(f"{text}")
    print(f"{len(tokens)} tokens:{tokens}")
    # show the pieces
    piece=[]
    for t in tokens:
        pieces=tokenizer.decode([t])
        piece.append(pieces)
    print(f"pieces:{piece}")
    print()

## 🧪 Try It Yourself

Tokenize your own text! Try:
- Your name
- A sentence in another language
- Some code
- Emojis

In [ ]:
my_text="my name is sidhhay"
tokens=tokenizer.encode(my_text)
print(f"my text ->{my_text}")
print(f"number of tokens ->{len(tokens)}")
print(f"encoded tokens:{tokens}")
pieces=[]
print(tokenizer.decode(tokens))
for t in tokens:
    #print([t])
    #print(t)
    pieces.append(tokenizer.decode([t]))

print(f"pieces:{pieces}")

## 1.2 Why Tokenization Matters

Token count affects:
- API costs (you pay per token)
- Context limits (GPT-4 has 128K token limit)
- Model behavior (some tasks break across token boundaries)
- model has context limits
If your text tokenizes inefficiently:
 You hit limit faster
 Model forgets earlier parts
 Better tokenization = more usable context.

⚡  It Affects Speed

Tokenization happens before every inference.

Slow tokenizer = slower API response.

That’s why tiktoken is optimized.
# Tokenization = the language the model speaks internally

In [15]:
# Compare token efficiency across different content types
test_cases = {
    "English prose": "The quick brown fox jumps over the lazy dog.",
    "Python code": "def hello():\n    print('Hello, world!')",
    "JSON": '{"name": "Alice", "age": 30, "city": "NYC"}',
    "Numbers": "1234567890 9876543210 1111111111",
    "URL": "https://www.example.com/path/to/page?query=value",
}
print("token efficency comparison:\n")
for name,text in test_cases.items():
    tokens=tokenizer.encode(text)
    chars=len(text)
    ratio=chars/len(tokens)
    print(f"{name}:")
    print(f" {chars} chars → {len(tokens)} tokens ({ratio:.1f} chars/token)")


token efficency comparison:

English prose:
 44 chars → 10 tokens (4.4 chars/token)
Python code:
 39 chars → 11 tokens (3.5 chars/token)
JSON:
 43 chars → 19 tokens (2.3 chars/token)
Numbers:
 32 chars → 14 tokens (2.3 chars/token)
URL:
 48 chars → 11 tokens (4.4 chars/token)


**Key insight**: Different content has different token efficiency.
- English prose: ~4 characters per token
- Code/JSON: often less efficient (more tokens per character)
- This affects your API costs!

## 1.3 The Token Boundary Problem

Some tasks are hard because they require reasoning WITHIN tokens.